# Spike V7: rapor modelleri eğitimi (Loco)

**Ne yapar:** `configs/v2_pretrain_v7.yaml` ile (kanal bozulması augmentation'ı, §12.19) rapor seed'leri 0–4 için
ön-eğitim + ince ayar + Day-0 modellerini eğitir (`--targets all --report-test`). Bu modeller V7 **test** koşusunun
girdisidir. Ayar (tune) modelleri yerelde eğitiliyor (`runs/chain_v7_tune.sh`), burada değil.

**Nereye yazar:** Drive'ım → `Spike_colab/runs/` (koşu klasörleri, checkpoint'ler, loglar). Bağlantı koparsa
defteri baştan çalıştırın: biten adımlar atlanır, yarım kalan adım `--resume` ile kaldığı yerden devam eder.

**Önce:** `notebooks/README.md` adım 1–3 (paketleme ve Drive'a yükleme). Çalışma zamanı: **GPU** (L4 önerilir).
Tarayıcı sekmesini açık tutun (Colab Pro'da arka planda çalışma yok).

Sırayla her hücreyi çalıştırın (Çalışma zamanı → Tümünü çalıştır da olur).

## 1. Ayarlar

In [ ]:
PRIMAT = "loco"                  # bu defter yalnız bu primatı eğitir; diğeri kendi defterinde
SEEDLER = [0, 1, 2, 3, 4]           # rapor seed'leri (KV8: ≥ 5 seed)
DRIVE_KLASOR = "/content/drive/MyDrive/Spike_colab"
DUMAN = False                       # True: 1 epoch'luk hızlı deneme (çıktı /content/duman, Drive'a yazılmaz)

## 2. GPU kontrolü

In [ ]:
!nvidia-smi
import sys, torch
print(sys.version)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU (L4)"

## 3. Drive'ı bağla ve paketi aç

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, shutil, zipfile, pathlib
D = pathlib.Path(DRIVE_KLASOR)
for f in ["spike_kod.zip", "spike_veri.zip"]:
    assert (D / f).exists(), f"{D / f} bulunamadı: notebooks/README.md adım 3"
KOD = pathlib.Path("/content/Spike")
if KOD.exists():
    shutil.rmtree(KOD)
KOD.mkdir()
for f in ["spike_kod.zip", "spike_veri.zip"]:
    with zipfile.ZipFile(D / f) as z:
        z.extractall(KOD)
os.chdir(KOD)
surum = json.loads((KOD / "KOD_SURUMU.json").read_text(encoding="utf-8"))
print("kod:", surum["commit"], "(kirli)" if surum["dirty"] else "(temiz)", "| paket:", surum["paket_zamani"])
print("önbellek:", sorted(p.name for p in (KOD / "data/primate/cache").iterdir()))

## 4. Bağımlılıklar ve hızlı test

Colab'da torch, numpy, pyyaml kurulu gelir. R² metriği için `neurobench` (yerel sürüm 2.3.0) kurulur.
Kurulum Colab'ın torch'unu değiştirmemeli; aşağıdaki kontrol bunu doğrular.

In [ ]:
import torch as _t
_onceki = _t.__version__
!pip -q install pyyaml numpy pytest neurobench==2.3.0
!python -c "import torch; from neurobench.metrics.workload import R2; print('neurobench tamam | torch', torch.__version__, '| CUDA', torch.cuda.is_available())"
print('kurulumdan önceki torch:', _onceki, '(üstteki satırla aynı olmalı; farklıysa Çalışma zamanı → Oturumu yeniden başlat)')
!python -m pytest tests/v2/test_train.py -q -x

## 5. Eğitim

Her seed için iki ayrı süreç (KR-043): önce `pretrain + finetune`, sonra `day0`. Çöken süreç en çok 2 kez
`--resume` ile yeniden denenir. Biten adımlar `runs/_bitti/` işaretiyle atlanır. Süre: seed başına ~40–90 dk
(GPU'ya göre); 5 seed birkaç saat.

In [ ]:
import glob, subprocess, time

LOG_DIR = "/content/duman" if DUMAN else str(D / "runs")
os.makedirs(LOG_DIR, exist_ok=True)
ISARET = pathlib.Path(LOG_DIR) / "_bitti"
ISARET.mkdir(exist_ok=True)
GRUPLAR = [("ptft", ["pretrain", "finetune"]), ("day0", ["day0"])]
ENV = {**os.environ, "PYTHONFAULTHANDLER": "1", "PYTHONIOENCODING": "utf-8"}


def klasorler(seed):
    """Bu primat ve seed'in koşu klasörleri, en yenisi önce (--resume havuzu)."""
    return sorted(glob.glob(f"{LOG_DIR}/*_v2_pretrain_v7_{PRIMAT}_seeds{seed}"), reverse=True)


def calistir(seed, etiket, asamalar):
    isaret = ISARET / f"{PRIMAT}_s{seed}_{etiket}"
    if isaret.exists():
        print(f"[atla] {PRIMAT} seed {seed} {etiket} (bitmiş)")
        return
    for deneme in (1, 2, 3):
        havuz = klasorler(seed)
        cmd = ["python", "experiments/v2_pretrain.py", "--config", "configs/v2_pretrain_v7.yaml",
               "--primates", PRIMAT, "--seeds", str(seed), "--stages", *asamalar,
               "--targets", "all", "--report-test", "--set", f"experiment.log_dir={LOG_DIR}"]
        if havuz:
            cmd += ["--resume", *havuz]
        if DUMAN:
            cmd += ["--epochs", "1"]
        log = pathlib.Path(LOG_DIR) / f"log_{PRIMAT}_s{seed}_{etiket}_{deneme}.txt"
        t0 = time.time()
        with open(log, "w", encoding="utf-8") as f:
            rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=ENV).returncode
        yeni = klasorler(seed)
        if yeni:                                              # KV8: Colab'da git yok, kod sürümü dosyası kopyalanır
            shutil.copy(KOD / "KOD_SURUMU.json", pathlib.Path(yeni[0]) / "KOD_SURUMU.json")
        print(f"{PRIMAT} seed {seed} {etiket} deneme {deneme}: çıkış {rc}, {(time.time() - t0) / 60:.1f} dk → "
              f"{yeni[0] if yeni else '-'}", flush=True)
        if rc == 0:
            isaret.touch()
            return
        print(log.read_text(encoding="utf-8")[-3000:])
    raise RuntimeError(f"{PRIMAT} seed {seed} {etiket}: 3 denemede de başarısız; log: {LOG_DIR}")


for s in SEEDLER:
    for etiket, asamalar in GRUPLAR:
        calistir(s, etiket, asamalar)
print("BİTTİ:", PRIMAT, SEEDLER)

## 6. Özet

In [ ]:
for s in SEEDLER:
    for k in klasorler(s):
        k = pathlib.Path(k)
        print(k.name, sorted(p.name for p in k.glob("*.pt")))
        t = k / "table.md"
        if t.exists():
            print(t.read_text(encoding="utf-8"))

## 7. Bitince

Drive'ım → `Spike_colab/runs/` klasörünün tamamını bilgisayarınızda `Spike/runs/colab_v7/` altına indirin
(`_bitti/` ve `log_*.txt` dahil). Sonra Claude'a "Colab eğitimi bitti" deyin; test koşusu bu klasörlerle yapılır.